# PRG4206 - Demo of the classifier inside Colab 
Run the cells top to bottom. No GPU needed. Takes about 1 minute.

In [ ]:
from google.colab import files
up = files.upload()          # choose mh_prototype_final.zip

In [ ]:
!rm -rf mh_project && mkdir mh_project
!unzip -oq mh_prototype_final.zip -d mh_project
%cd mh_project
!python src/06_train_app_model.py

## The prediction function (same logic as app.py)

In [ ]:
import re, joblib, numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

pipe = joblib.load("models/LogisticRegression.joblib")

def clean(text):
    words = re.sub(r"[^a-z\s]", " ", text.lower()).split()
    return " ".join(w for w in words if w not in ENGLISH_STOP_WORDS)

def analyse(text, top_k=8):
    prob = float(pipe.predict_proba([clean(text)])[0, 1])
    label = "SUICIDE-RISK-LIKE" if prob >= 0.5 else "DEPRESSION-LIKE"
    tfidf, clf = pipe.named_steps["tfidf"], pipe.named_steps["clf"]
    x = tfidf.transform([clean(text)])
    words = np.array(tfidf.get_feature_names_out())
    contrib = x.multiply(clf.coef_[0]).toarray()[0]
    order = np.argsort(np.abs(contrib))[::-1][:top_k]
    print(f"Prediction: {label}   (confidence {max(prob, 1-prob):.0%})")
    print("Words that influenced it (+ toward suicide-risk, - toward depression):")
    for i in order:
        if contrib[i] != 0:
            print(f"   {words[i]:<18} {contrib[i]:+.2f}")
    print("-" * 60)

## Test cases 

In [ ]:
tests = {
 "Test 1 - depression-like": "I have been feeling empty for months and I cannot enjoy anything anymore. Work is exhausting and I barely sleep.",
 "Test 2 - suicide-risk-like": "I do not see the point anymore. I keep thinking about ending it all and I have a plan.",
 "Test 3 - ambiguous": "Everything feels heavy lately, my friends say I should talk to someone about how low I feel.",
}
for name, text in tests.items():
    print(name); print(text); analyse(text)

## Try your own text

In [ ]:
text = input("Paste a post and press Enter: ")
analyse(text)